# 03 - Train Continuous ISL Sentence Recognition Model

Trains the **ST-GCN + Temporal Transformer + CTC** network architecture with Automatic Mixed Precision (AMP) on GPU / CPU.

The training pipeline runs a quick debug/verification batch first to confirm convergence and shape consistency before full-scale training.

In [ ]:
# 1. Environment & GPU Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Import Training Pipeline & Modules
import os, sys
from pathlib import Path

WORKING_DIR = Path("/kaggle/working/sentence-model-training") if os.path.exists("/kaggle") else Path("..")
sys.path.insert(0, str(WORKING_DIR.resolve()))

from config import CHECKPOINT_DIR, OUTPUT_DIR, VOCABULARY_FILE
from src.vocabulary import GlossVocabulary
from src.train import run_training
print("Training modules loaded successfully.")

In [ ]:
# 3. Run Tiny Debug Batch Verification (1 Epoch, Synthetic/Mini Subset)
print("--- Running Quick Smoke / Debug Batch ---")
run_training(
    dataset_path=None,
    annotations_path=None,
    checkpoint_dir=str(CHECKPOINT_DIR),
    num_epochs=1,
    batch_size=4,
    learning_rate=1e-4,
    debug=True,
)
print("Debug batch passed successfully!")

In [ ]:
# 4. Full Model Training (Set parameters and run when ready)
# Set DATASET_KEYPOINTS_DIR and ANNOTATIONS_CSV to your actual extracted dataset
DATASET_KEYPOINTS_DIR = Path("/kaggle/working/keypoints_cache") if os.path.exists("/kaggle") else Path("../keypoints_cache")
ANNOTATIONS_CSV = Path("/kaggle/input/isltranslate/annotations.csv") if os.path.exists("/kaggle") else Path("../outputs/sample_annotations.csv")

# To launch full training, set START_FULL_TRAIN = True
START_FULL_TRAIN = False

if START_FULL_TRAIN and DATASET_KEYPOINTS_DIR.exists() and ANNOTATIONS_CSV.exists():
    run_training(
        dataset_path=str(DATASET_KEYPOINTS_DIR),
        annotations_path=str(ANNOTATIONS_CSV),
        checkpoint_dir=str(CHECKPOINT_DIR),
        vocab_path=str(VOCABULARY_FILE),
        uid_col="uid",
        text_col="english",
        min_token_freq=1,
        num_epochs=35,
        batch_size=8,
        learning_rate=1e-4,
        debug=False,
    )
else:
    print("Full training skipped (START_FULL_TRAIN = False). Set to True to start training on the complete dataset.")